# AWS Glue Workshop — Module 05
## Notebook & Interactive Sessions

Use this notebook to explore the Module 1 Glue Data Catalog, transform data with PySpark, and write curated Parquet to S3.


In [ ]:
%idle_timeout 15

# Optional if the notebook UI has not already configured the role:
# %iam_role arn:aws:iam::ACCOUNT_ID:role/AWSGlueWorkshopRole
# %region us-east-1


In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print('Spark version:', spark.version)


In [ ]:
DATABASE = "glue_demo_db"
customers_df = glueContext.create_dynamic_frame.from_catalog(database=DATABASE, table_name="customers", transformation_ctx="customers_source").toDF()
products_df = glueContext.create_dynamic_frame.from_catalog(database=DATABASE, table_name="products", transformation_ctx="products_source").toDF()
orders_df = glueContext.create_dynamic_frame.from_catalog(database=DATABASE, table_name="orders", transformation_ctx="orders_source").toDF()

print("customers:", customers_df.count())
print("products :", products_df.count())
print("orders   :", orders_df.count())


In [ ]:
sales_df = (orders_df
    .join(customers_df, 'customer_id', 'inner')
    .join(products_df, 'product_id', 'inner'))

sales_df.select(
    'order_id','order_date','customer_id','customer_name','country',
    'product_id','product_name','category','quantity','order_amount','order_status'
).show(20, truncate=False)


In [ ]:
# Replace with the Module 1 bucket name.
OUTPUT_PATH = "s3://YOUR_WORKSHOP_BUCKET/curated/notebook-sales/"

sales_df.write.mode('overwrite').parquet(OUTPUT_PATH)
print('Written:', OUTPUT_PATH)
